# MLOps Lab 1 – Run scripts as a pipeline job

**Nakoba Hands-on Lab – Run pipelines in Azure Machine Learning**

A **pipeline** chains steps (here: *prepare data* → *train model*) so the whole workflow runs as one job, reproducibly, on a compute cluster. Each step is a reusable **component**: a script plus its inputs, outputs and environment.

```
diabetes-data (CSV)  ──►  [prep_data]  ──cleaned, normalised data──►  [train_model]  ──►  MLflow model
```

## Before you start

The cell below installs or updates the packages this notebook needs **into the notebook's own kernel**. (Installing with `pip` in the terminal can put them in a different Python environment, which is a common cause of `ModuleNotFoundError`.)

> If the output ends with *"you may need to restart the kernel"*, click **⟳ Restart** in the toolbar, then continue from the **next** cell.

In [ ]:
%pip install --quiet --upgrade azure-ai-ml azure-identity

## Connect to your workspace

On a compute instance, the workspace details are already configured, so `from_config()` finds them.

> If a banner asks you to **Authenticate**, click it and sign in, then run the cell again.

In [ ]:
from lab_helpers import get_ml_client, ensure_cluster, ensure_data_assets, get_sklearn_environment

ml_client = get_ml_client()
cluster_name = ensure_cluster(ml_client)   # creates 'aml-cluster' if it's missing
ensure_data_assets(ml_client)             # creates 'diabetes-data' if it's missing

## Create the scripts

Two scripts, one per pipeline step. Run both cells to write them into a **src** folder.

In [ ]:
import os

script_folder = "src"
os.makedirs(script_folder, exist_ok=True)
print(script_folder, "folder created")

In [ ]:
%%writefile $script_folder/prep-data.py
# Step 1: remove missing values and normalise the numeric columns
import argparse
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import MinMaxScaler


def main(args):
    df = pd.read_csv(args.input_data)
    print(f"Preparing {len(df)} rows of data")
    df = df.dropna()
    num_cols = ["Pregnancies", "PlasmaGlucose", "DiastolicBloodPressure", "TricepsThickness",
                "SerumInsulin", "BMI", "DiabetesPedigree"]
    df[num_cols] = MinMaxScaler().fit_transform(df[num_cols])
    Path(args.output_data).mkdir(parents=True, exist_ok=True)
    df.to_csv(Path(args.output_data) / "diabetes.csv", index=False)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_data", dest="input_data", type=str)
    parser.add_argument("--output_data", dest="output_data", type=str)
    return parser.parse_args()


if __name__ == "__main__":
    print("\n\n" + "*" * 60)
    main(parse_args())
    print("*" * 60 + "\n\n")

In [ ]:
%%writefile $script_folder/train-model.py
# Step 2: train a logistic regression model and save it as an MLflow model
import argparse
import glob

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split


def main(args):
    mlflow.autolog()
    files = glob.glob(args.training_data + "/*.csv")
    df = pd.concat((pd.read_csv(f) for f in files), sort=False)

    print("Splitting data...")
    X = df[["Pregnancies", "PlasmaGlucose", "DiastolicBloodPressure", "TricepsThickness",
            "SerumInsulin", "BMI", "DiabetesPedigree", "Age"]].values
    y = df["Diabetic"].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    mlflow.log_param("Regularization rate", args.reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1 / args.reg_rate, solver="liblinear").fit(X_train, y_train)
    mlflow.sklearn.save_model(model, args.model_output)

    acc = np.average(model.predict(X_test) == y_test)
    y_scores = model.predict_proba(X_test)
    print("Accuracy:", acc)
    print("AUC:", roc_auc_score(y_test, y_scores[:, 1]))

    fpr, tpr, _ = roc_curve(y_test, y_scores[:, 1])
    fig = plt.figure(figsize=(6, 4))
    plt.plot([0, 1], [0, 1], "k--")
    plt.plot(fpr, tpr)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.savefig("ROC-Curve.png")
    plt.close(fig)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--training_data", dest="training_data", type=str)
    parser.add_argument("--reg_rate", dest="reg_rate", type=float, default=0.01)
    parser.add_argument("--model_output", dest="model_output", type=str)
    return parser.parse_args()


if __name__ == "__main__":
    print("\n\n" + "*" * 60)
    main(parse_args())
    print("*" * 60 + "\n\n")

## Define the components

A **component** describes one step: its inputs, outputs, script and environment. The `get_sklearn_environment` helper picks a current curated scikit-learn environment, so the steps don't fail because an old environment was retired. The next cell writes and prints the two component YAML files.

In [ ]:
from pathlib import Path

sklearn_env = get_sklearn_environment(ml_client)

prep_yml = f"""$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: prep_data
display_name: Prepare training data
version: 1
type: command
inputs:
  input_data:
    type: uri_file
outputs:
  output_data:
    type: uri_folder
code: ./src
environment: {sklearn_env}
command: >-
  python prep-data.py
  --input_data ${{{{inputs.input_data}}}}
  --output_data ${{{{outputs.output_data}}}}
"""

train_yml = f"""$schema: https://azuremlschemas.azureedge.net/latest/commandComponent.schema.json
name: train_model
display_name: Train a logistic regression model
version: 1
type: command
inputs:
  training_data:
    type: uri_folder
  reg_rate:
    type: number
    default: 0.01
outputs:
  model_output:
    type: mlflow_model
code: ./src
environment: {sklearn_env}
command: >-
  python train-model.py
  --training_data ${{{{inputs.training_data}}}}
  --reg_rate ${{{{inputs.reg_rate}}}}
  --model_output ${{{{outputs.model_output}}}}
"""

Path("prep-data.yml").write_text(prep_yml, encoding="utf-8")
Path("train-model.yml").write_text(train_yml, encoding="utf-8")
print(prep_yml)
print(train_yml)

## Load the components

In [ ]:
from azure.ai.ml import load_component

prep_data = load_component(source="./prep-data.yml")
train_logistic_regression = load_component(source="./train-model.yml")

## Build the pipeline

The output of `prep_data` becomes the input of `train_model`: that's what links the two steps.

In [ ]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.dsl import pipeline


@pipeline()
def diabetes_classification(pipeline_job_input):
    clean_data = prep_data(input_data=pipeline_job_input)
    train_model = train_logistic_regression(training_data=clean_data.outputs.output_data)
    return {
        "pipeline_job_transformed_data": clean_data.outputs.output_data,
        "pipeline_job_trained_model": train_model.outputs.model_output,
    }


pipeline_job = diabetes_classification(Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:1"))
print(pipeline_job)

## Set where it runs and where outputs go

`upload` mode saves each output to storage, `default_compute` runs every step on the cluster, and `default_datastore` is where outputs are stored.

In [ ]:
pipeline_job.outputs.pipeline_job_transformed_data.mode = "upload"
pipeline_job.outputs.pipeline_job_trained_model.mode = "upload"
pipeline_job.settings.default_compute = cluster_name
pipeline_job.settings.default_datastore = "workspaceblobstore"
print(pipeline_job)

## Submit the pipeline

It takes about **15–20 minutes** (the cluster starts, the environment loads, then both steps run).

In [ ]:
pipeline_job = ml_client.jobs.create_or_update(pipeline_job, experiment_name="pipeline_diabetes")
print("Pipeline job name:", pipeline_job.name)
print("Monitor it at:", pipeline_job.studio_url)

## Review the pipeline in the studio

1. Click the link above (or **Jobs** → **pipeline_diabetes** → your job).
2. The **graph** shows both steps. Each turns green when it completes.
3. Click **Prepare training data** → **Outputs + logs** → **user_logs/std_log.txt**: *Preparing 10000 rows of data*.
4. Click **Train a logistic regression model** → **Metrics** (autologged by MLflow) and **Outputs + logs** (the saved model).

(Optional) Run the next cell after the pipeline completes.

In [ ]:
job = ml_client.jobs.get(pipeline_job.name)
print("Status:", job.status)